In [ ]:
import os, random, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import tensorflow as tf
tf.random.set_seed(SEED)
print("Tensorflow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

RESULTS = {}         # our running log for steps 8 and 15

In [ ]:
!pip -q install kagglehub
import kagglehub

root = Path(kagglehub.dataset_download("tawsifurrahman/covid19-radiography-database"))
print("Dataset folder:", root)
print("Contents:", [p.name for p in root.iterdir()])

In [ ]:
base = root / "COVID-19_Radiography_Dataset"
print("Subfolders:", [p.name for p in base.iterdir()])
print("Inside COVID:", [p.name for p in (base / "COVID").iterdir()])

# glob is case-insensitive-safe here: we search for the images folder whatever its capitalisation
def find_pngs(class_dir):
    img_dir = [p for p in class_dir.iterdir() if p.is_dir() and p.name.lower() == "images"][0]
    return sorted(img_dir.glob("*.png"))

covid_files  = find_pngs(base / "COVID")
normal_files = find_pngs(base / "Normal")
print("COVID images :", len(covid_files))
print("Normal images:", len(normal_files))
assert len(covid_files) > 0 and len(normal_files) > 0, "No files found: check the folder names printed above"

df = pd.DataFrame({
    "path":  [str(p) for p in covid_files + normal_files],
    "label": [1] * len(covid_files) + [0] * len(normal_files),   # 1 = COVID, 0 = Normal
})
df["class_name"] = df["label"].map({0: "Normal", 1: "COVID"})
print(df.class_name.value_counts())
RESULTS["step3_counts"] = df.class_name.value_counts().to_dict()
df.head()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14,7))
for r, cls in enumerate(["COVID", "Normal"]):
  sample = df[df.class_name == cls].sample(4, random_state=SEED)
  for c, p in enumerate(sample.path):
    axes[r, c].imshow(Image.open(p).convert("L"), cmap="gray")
    axes[r, c].set_title(cls); axes[r,c ].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
meta_dir = root / "COVID-19_Radiography_Dataset"

source_tables = {}
for name in ["COVID", "Normal"]:
    m = pd.read_excel(meta_dir / f"{name}.metadata.xlsx")
    print(f"\n===== {name}: {m.shape[0]} rows =====")
    print("Columns:", m.columns.tolist())
    src = m["URL"].str.extract(r"https?://(?:www\.)?([^/]+)")[0]   # keep only the website name
    source_tables[name] = src.value_counts()
    print(source_tables[name])

# Do the two classes share ANY source?
shared = set(source_tables["COVID"].index) & set(source_tables["Normal"].index)
print("\nSources shared by COVID and Normal:", shared if shared else "NONE")
RESULTS["step4b_shared_sources"] = list(shared)

In [ ]:
meta_dir = root / "COVID-19_Radiography_Dataset"          # defined here so this cell works on its own
readmes = sorted(meta_dir.glob("README*"))                  # the file is called README.md.txt, search instead of guessing
print("README files found:", [p.name for p in readmes])
if readmes:
    print(readmes[0].read_text(errors="ignore")[:3000])

In [ ]:
RESULTS["step1_counts"] = {"COVID": 3616, "Normal": 10192}
RESULTS["step1_source_overlap"] = "none: COVID = BIMCV/SIRM/Eurorad/GitHub, Normal = Kaggle (RSNA + pediatric)"

In [ ]:
def md5_of_file(p):
    with open(p, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

df["md5"] = df["path"].map(md5_of_file)
n_dup = int(df["md5"].duplicated().sum())
print("Exact duplicate files:", n_dup)

conflict = df.groupby("md5")["label"].nunique()
print("Hashes with conflicting labels:", int((conflict > 1).sum()))

print("Before dedup:", df.shape)
df = df.drop_duplicates("md5").reset_index(drop=True)
print("After dedup:", df.shape)
RESULTS["step2_duplicates"] = n_dup

In [ ]:
sizes, modes, bad = [], [], []
for p in df.sample(500, random_state=SEED).path:
    try:
        im = Image.open(p); im.verify()      # verify() checks the file is not broken...
        im = Image.open(p)                   # ...but closes it, so we reopen to read size/mode
        sizes.append(im.size); modes.append(im.mode)
    except Exception as e:
        bad.append(p)

print("Corrupt files in sample:", len(bad))
print("Image sizes:", pd.Series(sizes).value_counts().to_dict())    # to_dict() needs the ()
print("Color modes:", pd.Series(modes).value_counts().to_dict())

In [ ]:
def mean_intensity(p):
  return np.array(Image.open(p).convert("L"), dtype=np.float32).mean()

sample = df.groupby("label", group_keys=False).sample(200, random_state=SEED).copy()
sample["mean_pixel"] = sample.path.map(mean_intensity)
print(sample.groupby("class_name")["mean_pixel"].describe()[["mean", "std", "min", "max"]])


sample.boxplot(column="mean_pixel", by="class_name"); plt.suptitle(""); plt.show()

In [ ]:
RESULTS["step2_after_dedup"] = df.shape[0]
RESULTS["step2_format"] = "all 299x299, grayscale (L), 0 corrupt in sample of 500"
RESULTS["step2_brightness"] = "COVID median ~145 vs Normal ~125; COVID has dark outliers (padding)"

In [ ]:
from sklearn.model_selection import train_test_split

MAX_PER_CLASS = 1500         # lower it (e.g. 800) if training is too slow
n = min(MAX_PER_CLASS, df.label.value_counts().min())
df_small = df.groupby("label", group_keys=False).sample(n=n, random_state=SEED).reset_index(drop=True)

train_df, temp_df = train_test_split(df_small, test_size=0.30, stratify=df_small.label, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df.label, random_state=SEED)   # split temp_df, NOT df_small

# leakage guard: no identical file may appear in two splits (& = "in both")
assert not set(train_df.md5) & set(val_df.md5)
assert not set(train_df.md5) & set(test_df.md5)
assert not set(val_df.md5) & set(test_df.md5)      # was val & val (compared the set with itself)

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, len(d), d.label.mean().round(3))

RESULTS["step3_sizes"] = [len(train_df), len(val_df), len(test_df)]

In [ ]:
IMG, BATCH = 224, 32

def load_img(path, label):
  img = tf.io.read_file(path)
  img = tf.image.decode_png(img, channels=3)
  img = tf.image.resize(img, (IMG, IMG))          # float32, values 0..255
  return img, label

def make_ds(d, training=False):
  ds = tf.data.Dataset.from_tensor_slices((d.path.values, d.label.values.astype("float32")))
  ds = ds.map(load_img, num_parallel_calls=tf.data.AUTOTUNE)
  ds = ds.cache()                                # decoding each image once, then reuse from RAM
  if training:
    ds = ds.shuffle(1000, seed=SEED)
  return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds, val_ds, test_ds = make_ds(train_df, True), make_ds(val_df), make_ds(test_df)

xb, yb = next(iter(train_ds))
print("Batch shape:", xb.shape, "| labels:", yb.shape,
      "| pixel range:", float(tf.reduce_min(xb)), float(tf.reduce_max(xb)))

In [ ]:
RESULTS["step3_split"] = "balanced 1500/class, stratified 70/15/15, MD5 overlap = 0"

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

augment = keras.Sequential([
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10)
], name="augment")      # active only during training: no horizontal flip (it would swap heart side)

def build_cnn():
  inp = keras.Input((IMG, IMG, 3))
  x = augment(inp)
  x = layers.Rescaling(1 / 255.0)(x)
  for f in (16, 32, 64):
    x = layers.Conv2D(f, 3, activation="relu")(x)
    x = layers.MaxPooling2D()(x)
  x = layers.GlobalAveragePooling2D()(x)
  x = layers.Dropout(0.3)(x)
  out = layers.Dense(1, activation="sigmoid")(x)
  return keras.Model(inp, out, name="small_cnn")

cnn = build_cnn()
cnn.compile(optimizer=keras.optimizers.Adam(1e-3),
            loss="binary_crossentropy",
            metrics=["accuracy", keras.metrics.AUC(name="auc")])
cnn.summary()

In [ ]:
def plot_history(h, title):
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(h.history["loss"], label="train"); ax[0].plot(h.history["val_loss"], label="val")
    ax[0].set_title(title + " - loss"); ax[0].legend()
    ax[1].plot(h.history["auc"], label="train"); ax[1].plot(h.history["val_auc"], label="val")
    ax[1].set_title(title + " - AUC"); ax[1].legend()
    plt.show()

early_cnn = keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=3, restore_best_weights=True)
hist_cnn = cnn.fit(train_ds, validation_data=val_ds, epochs=8, callbacks=[early_cnn])
plot_history(hist_cnn, "Small CNN")
RESULTS["step4_cnn_best_val_auc"] = float(max(hist_cnn.history["val_auc"]))

In [ ]:
print(hist_cnn.history.keys())
for k, v in hist_cnn.history.items():
  print("\n", k, [round(x, 3) for x in v])

plot_history(hist_cnn, "Small CNN")

In [ ]:

SAVE_DIR = Path("artifacts")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print("Saving to:", SAVE_DIR)
print("GPU:", tf.config.list_physical_devices("GPU"))             # must NOT be []

In [ ]:
def count_params(m):
  tr = sum(int(np.prod(w.shape)) for w in m.trainable_weights)
  nt = sum(int(np.prod(w.shape)) for w in m.non_trainable_weights)
  print(f"Trainable: {tr:,} | Non-trainable: {nt:,}")

base = keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights="imagenet")
base.trainable = False                     # freeze: keep what it already knows

inp = keras.Input((IMG, IMG, 3))
x = augment(inp)
x = keras.applications.mobilenet_v2.preprocess_input(x)       # scales 0..255 -> -1..1
x = base(x, training=False)                                   # keep BatchNorm layers in inference mode
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(1, activation="sigmoid")(x)
tl = keras.Model(inp, out, name="mobilenet_tl")

tl.compile(optimizer=keras.optimizers.Adam(1e-3),
           loss ="binary_crossentropy",
           metrics=["accuracy", keras.metrics.AUC(name="auc")])
count_params(tl)

early = keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=3, restore_best_weights=True)
hist_tl = tl.fit(train_ds, validation_data=val_ds, epochs=8, callbacks=[early])
plot_history(hist_tl, "Transfer learning (frozen)")

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:
  layer.trainable= False                                  # keep early layers (edges, textures) frozen

tl.compile(optimizer=keras.optimizers.Adam(1e-5),         # 100x smaller learning rate
           loss="binary_crossentropy",
           metrics=["accuracy", keras.metrics.AUC(name="auc")])
count_params(tl)


hist_ft = tl.fit(train_ds, validation_data=val_ds, epochs=6, callbacks=[early])
plot_history(hist_ft, "Fine-tuning")

In [ ]:
tl.save(SAVE_DIR / "covid_xray_mobilenet.keras")
train_df.to_csv(SAVE_DIR / "train_df.csv", index=False)
val_df.to_csv(SAVE_DIR / "val_df.csv", index=False)
test_df.to_csv(SAVE_DIR / "test_df.csv", index=False)

print("Best val_auc frozen    :", round(max(hist_tl.history['val_auc']), 4))
print("Best val_auc fine-tune :", round(max(hist_ft.history['val_auc']), 4))
print("Saved:", [p.name for p in SAVE_DIR.iterdir()])

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
import json
import numpy as np, pandas as pd
from pathlib import Path
if "root" not in globals():          # after a restart the dataset path is lost, so rebuild it
    import kagglehub
    root = Path(kagglehub.dataset_download("tawsifurrahman/covid19-radiography-database"))
SAVE_DIR = Path("artifacts")

train_df = pd.read_csv(SAVE_DIR / "train_df.csv")
val_df   = pd.read_csv(SAVE_DIR / "val_df.csv")
test_df  = pd.read_csv(SAVE_DIR / "test_df.csv")

# the dataset folder can change between sessions: rebuild each path from the new 'root'
def fix_path(p):
  rel = p.split("COVID-19_Radiography_Dataset/")[1]
  return str(root / "COVID-19_Radiography_Dataset" / rel)

for d in (train_df, val_df, test_df):
  d["path"] = d["path"].map(fix_path)
assert all(Path(p).exists() for p in test_df.path), "Some test images were not found"

tl = keras.models.load_model(str(SAVE_DIR / "covid_xray_mobilenet.keras"))
print("Model loaded:", tl.name, "| test images", len(test_df), "| COVID share:", test_df.label.mean())

In [ ]:
IMG, BATCH = 224, 32

def load_img(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, (IMG, IMG))          # float32, values 0..255
    return img, label

def make_ds(d):
    ds = tf.data.Dataset.from_tensor_slices((d.path.values, d.label.values.astype("float32")))
    return ds.map(load_img, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE)   # NOT shuffled

val_ds, test_ds = make_ds(val_df), make_ds(test_df)
print(tl.evaluate(val_ds, verbose=0, return_dict=True))

In [ ]:
aug = tl.get_layer("augment")
img = load_img(train_df.path.iloc[0], 0)[0]         # one training image, 224x224x3, values 0..255

fig, ax = plt.subplots(1, 6, figsize=(18, 3.5))
ax[0].imshow(img.numpy().astype("uint8"));ax[0].set_title("Original")
for i in range(1, 6):
  a = aug(img[None], training=True)[0]              # training=True switches augmentation ON
  ax[i].imshow(np.clip(a.numpy(), 0, 255).astype("uint8")); ax[i].set_title(f"Augmented {i}")
for a_ in ax: a_.axis("off")
plt.show()

In [ ]:
base = next(l for l in tl.layers if "mobilenet" in l.name)
names = ["Conv1_relu", "block_3_expand_relu", "block_13_expand_relu", "out_relu"]
viewer = keras.Model(base.inputs, [base.get_layer(n).output for n in names])

x = keras.applications.mobilenet_v2.preprocess_input(img[None])
maps = [m.numpy() for m in viewer(x, training=False)]

fig, ax =  plt.subplots(len(names), 7, figsize=(17, 10))
for r, (n, m) in enumerate(zip(names, maps)):
  ax[r, 0].imshow(img.numpy().astype("uint8")); ax[r, 0].set_title("Input", fontsize=9)
  top = np.argsort(m[0].mean(axis=(0, 1)))[::-1][:6]               # the 6 most active filters
  for c, ch in enumerate(top, start=1):
    ax[r, c].imshow(m[0, :, :, ch], cmap="viridis")
    ax[r, c].set_title(f"{n}\n{m.shape[1]}x{m.shape[2]}, filter {ch}", fontsize=8)
  for a_ in ax[r]: a_.axis("off")            # moved out of the inner loop: hide axes once per row
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.metrics import (confusion_matrix, classification_report, roc_auc_score,
                             average_precision_score, roc_curve, ConfusionMatrixDisplay)

y_val, y_test = val_df.label.values, test_df.label.values
p_val = tl.predict(val_ds, verbose=0).ravel()
p_test = tl.predict(test_ds, verbose=0).ravel()          # test_ds is NOT shuffled: order matches test_df

print("VAL  ROC-AUC :", round(roc_auc_score(y_val, p_val), 4))
print("TEST ROC-AUC :", round(roc_auc_score(y_test, p_test), 4))
print("TEST PR-AUC  :", round(average_precision_score(y_test, p_test), 4))

plt.figure(figsize=(9, 4))
plt.hist(p_test[y_test == 0], bins=30, alpha=0.6, label="Truly Normal")
plt.hist(p_test[y_test == 1], bins=30, alpha=0.6, label="Truly COVID")
plt.xlabel("Predicted probability of COVID"); plt.ylabel("Number of test images")
plt.title("Test set: how confident is the model ? :"); plt.legend(); plt.show()

In [ ]:
fpr, tpr, thr = roc_curve(y_val, p_val)
i = np.where(tpr >= 0.95)[0][0]
chosen = thr[i]
print("Chosen threshold : ", round(float(chosen), 4))

pred = (p_test >= chosen).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
sens, spec = tp / (tp + fn), tn / (tn + fp)
print(f"TP={tp} FP={fp} TN={tn} FN={fn}")
print(f"Sensitivity (COVID caught) : {round(sens, 4)}")
print(f"Specificity (Normal correctly cleared) : {round(spec, 4)}")
print(f"Precision (flagged COVID that really are) :  {round(tp / (tp + fp), 4)}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(fpr, tpr, label="model"); ax[0].plot([0, 1], [0, 1], "--", c="gray", label="random guess")
ax[0].plot(fpr[i], tpr[i],"ro", label=f"chosen threshold {chosen:.2f}")
ax[0].set_xlabel("False alarm rate (1 - specificity)"); ax[0].legend()
ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["Normal", "COVID"], cmap="Blues", ax=ax[1])
ax[1].set_title("Test confusion matrix"); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(p_test[y_test == 0], bins=30, alpha=0.6, label="Truly Normal")
plt.hist(p_test[y_test == 1], bins=30, alpha=0.6, label="Truly COVID")
plt.axvline(chosen, color="red", lw=2, label=f"threshold = {chosen:.2f}")
plt.axvspan(chosen, 1, color="red", alpha=0.07)
plt.text(chosen + 0.02, plt.ylim()[1] * 0.9, "flagged as COVID ->", color="red")
plt.xlabel("Predicted probability of COVID"); plt.ylabel("Number of test images")
plt.title("Everything right of the red line is flagged"); plt.legend(); plt.show()

In [ ]:
prevs = [0.5, 0.10, 0.05, 0.01]
ppvs = [(sens * p) / (sens * p + ( 1 - spec) * (1 - p)) for p in prevs]
for p, v in zip(prevs, ppvs):
  print(f"Prevalence {p:>4.0%} - precision = {v:.3f}")

plt.figure(figsize=(10, 5))
bars = plt.bar([f"{p:.0%}" for p in prevs], ppvs, color=["#4c72b0", "#dd8452", "#dd8452", "#c44e52"])
plt.bar_label(bars, fmt="%.2f"); plt.ylim(0, 1.05)
plt.xlabel("Share of patients who really have COVID"); plt.ylabel("Precision")
plt.title("Same model, same skill: precision collapses when COVID is rare"); plt.show()

In [ ]:
res = test_df.assign(prob=p_test, pred=pred)
fn_df = res[(res.label == 1) & (res.pred == 0)].sort_values("prob")                               # most confident misses first
fp_df = res[(res.label == 0) & (res.pred == 1)].sort_values("prob", ascending=False)             # most confident false alarms first
print("Missed COVID : ", len(fn_df), "| False alarms : ", len(fp_df))

fig, ax = plt.subplots(2, 5, figsize=(17, 7))
for r, (title, d) in enumerate([("Missed COVID (FN)", fn_df), ("False alarm  (FP)", fp_df)]):
  for c in range(5):
    ax[r, c].axis("off")
    if c < len(d):
      row = d.iloc[c]
      ax[r, c].imshow(Image.open(row.path).convert("L"), cmap="gray")
      ax[r, c].set_title(f"{title}\nP(COVID) = {row.prob:.2f}", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
RESULTS.update({
    "step1_counts": {"COVID": 3616, "Normal": 10192},
    "step1_source_overlap": "none: COVID = BIMCV/SIRM/Eurorad/GitHub, Normal = Kaggle(RSNA + pediatric)",
    "step2_after_dedup": 13761,
    "step4_cnn_best_val_auc": 0.7225,
    "step5_frozen_best_val_auc": 0.9528,
    "step5_finetune_best_val_auc": 0.9700,
    "step6_test_roc_auc": float(roc_auc_score(y_test, p_test)),
    "step6_test_pr_auc": float(average_precision_score(y_test, p_test)),
    "step6_threshold": float(chosen),
    "step6_sensitivity": float(sens),
    "step6_specificity": float(spec),
    "step6_confusion": {"TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn)},
})
with open(SAVE_DIR / "results.json", "w") as f:
  json.dump(RESULTS, f, indent=2)
print(json.dumps(RESULTS, indent=2))

In [ ]:
X_test = np.concatenate([xb.numpy() for xb, _ in test_ds])          # (450, 224, 224, 3)
print("Test array:", X_test.shape)

def auc_of(X, name):
  p = tl.predict(X, batch_size=BATCH, verbose=0).ravel()
  a = roc_auc_score(y_test, p)
  print(f"{name:<30} ROC-AUC = {a:.4f}")
  return a

lo, hi = 56, 168                                 # center 112x112 box, covers most of the lungs
X_no_center = X_test.copy(); X_no_center[:, lo:hi, lo:hi, :] = 0
X_only_center = np.zeros_like(X_test); X_only_center[:, lo:hi, lo:hi, :] = X_test[:, lo:hi, lo:hi, :]
X_corner = X_test.copy();          X_corner[:, 0:112, 0:112, :] = 0       # control: same-size block, NOT the lungs

aucs = {
    "Full image":                      auc_of(X_test, "Full Image"),
    "Lungs hidden":                    auc_of(X_no_center, "Lungs hidden"),
    "Lungs only":                      auc_of(X_only_center, "Lungs only"),
    "Corner hidden (control)":         auc_of(X_corner, "Corner Hidden (control)"),
}

plt.figure(figsize=(8, 4))
bars = plt.bar(list(aucs.keys()), list(aucs.values()), color=["#4c72b0", "#c44e52", "#55a868", "#8172b2"])
plt.bar_label(bars, fmt="%.3f"); plt.axhline(0.5, ls="--", c="gray", label="random guess (0.5)")
plt.ylim(0, 1.05); plt.ylabel("Test ROC-AUC"); plt.xticks(rotation=15); plt.legend()
plt.title("What happens when we hide parts of the X-ray?"); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, im, t in zip(ax, [X_test[0], X_no_center[0], X_only_center[0], X_corner[0]],
                    ["Full", "Lungs hidden", "Lungs only", "Corner hidden (control)"]):
  a.imshow(im.astype("uint8")); a.set_title(t); a.axis("off")
plt.show()

In [ ]:
base = next(l for l in tl.layers if "mobilenet" in l.name)
dense_layer = tl.layers[-1]                          # the final Dense(1, sigmoid)

def gradcam(img):
  x = keras.applications.mobilenet_v2.preprocess_input(tf.cast(img[None], tf.float32))
  with tf.GradientTape() as tape:
    conv = base(x, training=False)               # 7x7x1280 feature maps
    tape.watch(conv)
    prob = dense_layer(tf.reduce_mean(conv, axis=(1, 2)))        # same as pooling + head
  grads = tape.gradient(prob, conv)                 # how much each map pushed toward "COVID"
  weights = tf.reduce_mean(grads, axis=(1, 2))
  cam = tf.nn.relu(tf.reduce_sum(conv * weights[:, None, None, :], axis=-1))[0]
  cam = cam / (tf.reduce_max(cam) + 1e-8)
  cam = tf.image.resize(cam[..., None], (IMG, IMG))[..., 0].numpy()
  return cam, float(prob[0, 0])

# checkpoint: Grad-CAM's probability must match the model's own prediction
c, p = gradcam(X_test[0]); print("Grad-CAM prob : ", round(p, 4), "| model prob:", round(float(p_test[0]), 4))

groups = {
    "Correct COVID (TP)": res[(res.label == 1) & (res.pred == 1)].sort_values("prob", ascending=False),
    "Correct Normal (TN)": res[(res.label == 0) & (res.pred == 0)].sort_values("prob"),
    "Missed COVID (FN)": fn_df,
    "False Alarm (FP)": fp_df,
}
fig, ax = plt.subplots(4, 4, figsize=(15, 15))
for r, (title, d) in enumerate(groups.items()):
  for c in range(4):
    ax[r, c].axis("off")
    if c < len(d):
      row = d.iloc[c]
      im = load_img(row.path, 0)[0].numpy()
      cam, p = gradcam(im)
      ax[r, c].imshow(im.astype("uint8"))
      ax[r, c].imshow(cam, cmap="jet", alpha=0.4)
      ax[r, c].set_title(f"{title}\nP(COVID) = {p:.2f}", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
RESULTS["step7_masking_auc"] = {k: round(float(v), 4) for k, v in aucs.items()}
with open(SAVE_DIR / "results.json", "w") as f:
  json.dump(RESULTS, f, indent=2)
print(RESULTS["step7_masking_auc"])

In [ ]:
RESULTS = json.load(open(SAVE_DIR / "results.json"))     # reload the saved numbers

card = {
    "task": "Binary classification COVID vs Normal chest X-ray. EDUCATIONAL ONLY, not a diagnostic tool.",
    "data": "COVID-19 Radiography Database (Kaggle). Deduplicated by MD5 (47 removed), balanced subsample "
            "1,500 per class, stratified split 70/15/15.",
    "model": "MobileNetV2 pretrained on ImageNet, new sigmoid head, last 30 layers fine-tuned (lr 1e-5).",
    "results": {
        "val_auc_small_cnn": RESULTS["step4_cnn_best_val_auc"],
        "val_auc_finetuned": RESULTS["step5_finetune_best_val_auc"],
        "test_roc_auc": round(RESULTS["step6_test_roc_auc"], 4),
        "test_pr_auc": round(RESULTS["step6_test_pr_auc"], 4),
        "threshold_chosen_on_validation": round(RESULTS["step6_threshold"], 4),
        "test_sensitivity": round(RESULTS["step6_sensitivity"], 4),
        "test_specificity": round(RESULTS["step6_specificity"], 4),
    },
    "shortcut_audit": RESULTS["step7_masking_auc"],
    "limitations": [
        "Source confounding: COVID and Normal images come from entirely different sources (zero overlap).",
        "Age confounding: the Normal class includes pediatric X-rays; a pediatric COVID image was missed (P=0.01).",
        "Shortcut evidence: AUC stays at ~0.80 with the central lung region masked.",
        "No patient IDs: the same patient may appear in several splits.",
        "Balanced test set: at 1% real prevalence, precision would be ~4%.",
        "Threshold chosen for 95% sensitivity on validation gave ~91% on test.",
        "Single random split of 450 test images; no external validation.",
    ],
}
with open(SAVE_DIR / "model_card.json", "w") as f:
    json.dump(card, f, indent=2)
print(json.dumps(card, indent=2))

In [ ]:
R = RESULTS
s, sp = R["step6_sensitivity"], R["step6_specificity"]
fig, ax = plt.subplots(2, 2, figsize=(13, 10))

# 1. model performance
names = ["Small CNN\n(val)", "Frozen\n(val)", "Fine-tuned\n(val)", "Fine-tuned\n(test)"]
vals = [R["step4_cnn_best_val_auc"], R["step5_frozen_best_val_auc"],
        R["step5_finetune_best_val_auc"], R["step6_test_roc_auc"]
]
b = ax[0, 0].bar(names, vals, color=["#8c8c8c", "#4c72b0", "#4c72b0", "#dd8452"])
ax[0, 0].bar_label(b, fmt="%.3f"); ax[0, 0].set_ylim(0.5, 1.02)
ax[0, 0].set_title("1. Model performance (ROC-AUC)")

# 2. confusion matrix
c = R["step6_confusion"]
m = np.array([[c["TN"], c["FP"]], [c["FN"], c["TP"]]])
ax[0, 1].imshow(m, cmap="Blues")
for i in range(2):
  for j in range(2):
    ax[0, 1].text(j, i, m[i, j], ha="center", va="center", fontsize=16,
                  color="white" if m[i, j] > m.max() / 2 else "black")
ax[0, 1].set_xticks([0, 1]); ax[0, 1].set_xticklabels(["Normal", "COVID"])
ax[0, 1].set_yticks([0, 1]); ax[0, 1].set_yticklabels(["Normal", "COVID"])
ax[0, 1].set_xlabel("Predicted"); ax[0, 1].set_ylabel("True")
ax[0, 1].set_title(f"2. Test results (threshold {R['step6_threshold']:.2f})\n"
                    f"Sensitivity {s:.1%} | Specificity {sp:.1%}")

# 3. shortcut audit
mk = R["step7_masking_auc"]
b = ax[1, 0].bar(list(mk.keys()), list(mk.values()), color=["#4c72b0", "#c44e52", "#55a868", "#8172b2"])
ax[1, 0].bar_label(b, fmt="%.3f"); ax[1, 0].axhline(0.5, ls="--", c="gray")
ax[1, 0].set_title("3. Shortcut audit: AUC when parts are hidden")
ax[1, 0].set_xticklabels(list(mk.keys()), rotation=15)

# 4. precision at real-world prevalence
prevs = [0.5, 0.10, 0.05, 0.01]
ppv = [(s * p) / (s * p + (1 - sp) * (1 - p)) for p in prevs]
b = ax[1, 1].bar([f"{p:.0%}" for p in prevs], ppv, color=["#4c72b0", "#dd8452", "#dd8452", "#c44e52"])
ax[1, 1].bar_label(b, fmt="%.2f"); ax[1, 1].set_ylim(0, 1.05)
ax[1, 1].set_xlabel("Share of patients with COVID")
ax[1, 1].set_title("4. Precision at real-world prevalence")

fig.suptitle("COVID-19 chest X-ray classifier: educational audit (NOT a diagnostic tool)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(SAVE_DIR / "summary_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
low, high = 0.05, 0.90       # fixed in adavance for illustation, NOT tuned on test
zone = np.where(p_test < low, "1. auto-clear (Normal)",
               np.where(p_test >= high, "3. auto-flag (COVID)", "2. human review"))
print(pd.crosstab(zone, np.where(y_test == 1, "truly COVID", "truly NORMAL"), margins=True))

plt.figure(figsize=(9, 4))
plt.axvspan(0, low, color="green", alpha=0.10, label="auto-clear")
plt.axvspan(low, high, color="orange", alpha=0.10, label="human review")
plt.axvspan(high, 1, color="red", alpha=0.10, label="auto-flag")
plt.hist(p_test[y_test == 0], bins=40, alpha=0.6, label="Truly Normal")
plt.hist(p_test[y_test == 1], bins=40, alpha=0.6, label="Truly COVID")
plt.xlabel("Predicted probability of COVID"); plt.ylabel("Number of test images")
plt.title("Triage: the model decides when it's very confident"); plt.legend(fontsize=8)
plt.savefig(SAVE_DIR / "triage_zones.png", dpi=150, bbox_inches="tight"); plt.show()